In [42]:
import os
from typing import List
from langchain_community.document_loaders import PyPDFLoader #, Docx2txtLoader, TextLoader
from langchain_core.documents import Document

def load_documents(directory_path: str) -> List[Document]:
    """
    NODO 1: Multi-Format Document Loader
    Soporta PDF, DOCX y TXT.
    """
    print(f"[LOADER] Iniciando escaneo en: {directory_path}")
    
    if not os.path.exists(directory_path):
        print(f"[LOADER] Error: El directorio no existe.")
        return []

    documents = []
    
    LOADER_MAPPING = {
        ".pdf": PyPDFLoader,
        #".docx": Docx2txtLoader,
        #".txt": TextLoader
    }

    files = os.listdir(directory_path)
    supported_files = [f for f in files if os.path.splitext(f)[1].lower() in LOADER_MAPPING]

    if not supported_files:
        print("[LOADER] No se encontraron archivos compatibles (PDF, DOCX, TXT).")
        return []

    print(f"[LOADER] Se encontraron {len(supported_files)} archivos compatibles.")

    for filename in supported_files:
        file_path = os.path.join(directory_path, filename)
        ext = os.path.splitext(filename)[1].lower()
        
        try:
            print(f"   └── Cargando ({ext}): {filename}...")
            
            loader_class = LOADER_MAPPING[ext]
            
            if ext == ".txt":
                loader = loader_class(file_path, encoding="utf-8")
            else:
                loader = loader_class(file_path)
                
            docs = loader.load()
            
            for doc in docs:
                doc.metadata["source_filename"] = filename
                doc.metadata["file_type"] = ext
            
            documents.extend(docs)
            print(f"       Éxito: {len(docs)} fragmentos/páginas extraídas.")
            
        except Exception as e:
            print(f"       Error cargando {filename}: {str(e)}")

    print(f"[LOADER] Finalizado. Total de documentos en memoria: {len(documents)}")
    return documents

In [43]:
documents = load_documents('/home/huascar/KANTUTA_AI/backend/data/raw')

[LOADER] Iniciando escaneo en: /home/huascar/KANTUTA_AI/backend/data/raw
[LOADER] Se encontraron 10 archivos compatibles.
   └── Cargando (.pdf): Ley 054 Ley de Protección legal de Niñas, Niños y Adolescentes.pdf...
       Éxito: 6 fragmentos/páginas extraídas.
   └── Cargando (.pdf): LEY 1173 DE ABREVIACIÓN PROCESAL PENAL Y DE FORTALECIMIENTO DE LA LUCHA INTEGRAL CONTRA LA VIOLENCIA A NIÑAS NIÑOS ADOLESCENTS Y MUJERES.pdf...
       Éxito: 64 fragmentos/páginas extraídas.
   └── Cargando (.pdf): Ley 065 Ley de Pensiones Bolivia.pdf...
       Éxito: 44 fragmentos/páginas extraídas.
   └── Cargando (.pdf): Ley393ServiciosFinancieros.pdf...
       Éxito: 167 fragmentos/páginas extraídas.
   └── Cargando (.pdf): Ley 045 LEY CONTRA EL RACISMO Y TODA FORMA DE DISCRIMINACIÓN.pdf...
       Éxito: 15 fragmentos/páginas extraídas.
   └── Cargando (.pdf): ley 263 Ley Integral contra la trata y tráfico de personas.pdf...
       Éxito: 29 fragmentos/páginas extraídas.
   └── Cargando (.pdf): LEY N

In [44]:
def load_single_document(file_path: str) -> List[Document]:
    """
    NODO 1 (SINGLE FILE): Carga un único documento específico.
    Soporta: PDF, DOCX, TXT.
    
    Args:
        file_path (str): Ruta absoluta o relativa al archivo específico.
        
    Returns:
        List[Document]: Lista de páginas/fragmentos del archivo cargado.
    """
    print(f"[LOADER] Intentando cargar archivo: {file_path}")
    
    # 1. Validar existencia
    if not os.path.exists(file_path):
        print(f"[LOADER] Error: El archivo no existe en la ruta indicada.")
        return []
    
    # 2. Validar extensión y seleccionar Loader
    ext = os.path.splitext(file_path)[1].lower()
    filename = os.path.basename(file_path)
    
    LOADER_MAPPING = {
        ".pdf": PyPDFLoader,
        #".docx": Docx2txtLoader,
        #".txt": TextLoader
    }
    
    if ext not in LOADER_MAPPING:
        print(f"[LOADER] Formato no soportado ({ext}). Solo se acepta PDF, DOCX o TXT.")
        return []

    try:
        # 3. Carga
        loader_class = LOADER_MAPPING[ext]
        
        if ext == ".txt":
            loader = loader_class(file_path, encoding="utf-8")
        else:
            loader = loader_class(file_path)
            
        docs = loader.load()
        
        # 4. Enriquecer Metadatos
        for doc in docs:
            doc.metadata["source_filename"] = filename
            doc.metadata["file_type"] = ext
            # Limpiamos la ruta para seguridad (no exponer estructura interna)
            doc.metadata["source"] = filename 

        print(f"   Éxito: Se extrajeron {len(docs)} páginas/bloques de '{filename}'.")
        return docs

    except Exception as e:
        print(f"   [LOADER] Error crítico procesando el archivo: {str(e)}")
        return []

In [45]:
document = load_single_document('/home/huascar/KANTUTA_AI/backend/data/raw/1-ley-348-2016.pdf')

[LOADER] Intentando cargar archivo: /home/huascar/KANTUTA_AI/backend/data/raw/1-ley-348-2016.pdf
   Éxito: Se extrajeron 112 páginas/bloques de '1-ley-348-2016.pdf'.


In [46]:
document[10]

Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2017-06-23T18:32:32-04:00', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'moddate': '2017-06-23T19:05:41-04:00', 'title': 'COMPENDIO SALUD 01.indd', 'trapped': '/False', 'source': '1-ley-348-2016.pdf', 'total_pages': 112, 'page': 10, 'page_label': '11', 'source_filename': '1-ley-348-2016.pdf', 'file_type': '.pdf'}, page_content='Ley 348 y reglamento\n11\nIV. Las disposiciones de la presente Ley serán aplicables a toda persona \nque por su situación de vulnerabilidad, sufra cualquiera de las formas \nde violencia que esta Ley sanciona, independientemente de su género.\nARTÍCULO 6. (DEFINICIONES). Para efectos de la aplicación \ne interpretación de la presente Ley, se adoptan las siguientes \ndefiniciones:\n1. Violencia. Constituye cualquier acción u omisión, abierta o \nencubierta, que cause la muerte, sufrimiento o daño físico, sexual \n

In [47]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents: List[Document], chunk_size: int = 1500, chunk_overlap: int = 200) -> List[Document]:
    """
    NODO 2: Text Splitter (Legal Semantic Chunking)
    Divide documentos grandes en fragmentos manejables respetando la estructura legal/académica.
    
    Args:
        documents: Lista de documentos crudos (salida del Nodo 1).
        chunk_size: Tamaño objetivo del fragmento (caracteres). 1500 es bueno para leyes.
        chunk_overlap: Solapamiento para mantener contexto entre cortes.
        
    Returns:
        List[Document]: Lista de chunks listos para embedding.
    """
    print(f"[SPLITTER] Iniciando fragmentación de {len(documents)} documentos base...")
    
    separators = [
        "\nLIBRO ",         # Libros de códigos
        "\nTÍTULO ",        # Títulos
        "\nCAPÍTULO ",      # Capítulos
        "\nARTÍCULO ",      # Artículos
        "\nArtículo ",
        "\nArt. ",          
        "\n\n",             # Párrafos dobles
        "\n",               # Párrafos simples
        ". ",               # Oraciones
        "\u200b",           # Zero-width space
        "\uff0c",           # Fullwidth comma
        "\u3001",           # Ideographic comma
        "\uff0e",           # Fullwidth full stop
        "\u3002",           # Ideographic full stop
        " ",                # Palabras
        ""                  # Caracteres (último recurso)
    ]
    
    text_splitter = RecursiveCharacterTextSplitter(
        separators=separators,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False, # No hay expresiones regulares complejas
        strip_whitespace=True
    )
    
    chunks = text_splitter.split_documents(documents)
    
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_index"] = i
        
    print(f"[SPLITTER] Generados {len(chunks)} fragmentos a partir de los documentos originales.")
    
    return chunks

In [48]:
chunks:List[Document] = split_documents(document)

[SPLITTER] Iniciando fragmentación de 112 documentos base...
[SPLITTER] Generados 164 fragmentos a partir de los documentos originales.


In [49]:
chunks[30:40]

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2017-06-23T18:32:32-04:00', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'moddate': '2017-06-23T19:05:41-04:00', 'title': 'COMPENDIO SALUD 01.indd', 'trapped': '/False', 'source': '1-ley-348-2016.pdf', 'total_pages': 112, 'page': 22, 'page_label': '23', 'source_filename': '1-ley-348-2016.pdf', 'file_type': '.pdf', 'chunk_index': 30}, page_content='ARTÍCULO 18. (PREVENCIÓN COMUNITARIA). Las autoridades indígena \noriginario campesinas y afrobolivianas, adoptarán en las comunidades \nen las que ejercen sus funciones, las medidas de prevención que \nconsideren más adecuadas bajo los tres criterios de acción establecidos \npara evitar todo acto de violencia hacia las mujeres, con la participación \nde éstas en su planificación, ejecución y seguimiento, respetando sus \nderechos. Ninguna norma o procedimiento propio de las naciones y \npuebl

In [50]:
import os
from langchain_ollama import OllamaEmbeddings 

def get_embedding_model():
    """
    NODO 3: Embedding Model Factory
    Retorna una instancia configurada del modelo de embeddings.
    Centraliza la configuración para evitar desajustes entre Ingesta y Consulta.
    """
    
    ollama_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11435")
    
    print(f"[EMBEDDING] Conectando a Ollama en: {ollama_url}")
    print(f"[EMBEDDING] Modelo seleccionado: qwen3-embedding:8b")

    embeddings = OllamaEmbeddings(
        model="qwen3-embedding:8b",
        base_url=ollama_url,
    )
    
    return embeddings

In [54]:
from typing import List
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

async def ingest_to_chroma(chunks: List[Document], embedding_model: Embeddings, reset_db: bool = False):
    """
    NODO 4 (SERVER MODE): Ingesta hacia ChromaDB corriendo en Docker.
    """
    
    print(f"[VECTOR STORE] Conectando a ChromaDB Server (localhost:8000)...")
    
    # 1. Inicialisar vector_store 
    collection_name = "kantuta_legal"
    
    client = Chroma(
        collection_name= collection_name,
        embedding_function= embedding_model,
        host= "localhost",
        port= 8000
    )

    # 2. Entrar a la coleccion

    # 3. Limpieza (Reset)
    if reset_db:
        try:
            print(f"[VECTOR STORE] Eliminando colección '{collection_name}'...")
            client.reset_collection()
        except Exception:
            print(f"La colección no existía, continuando...")

    print(f"[VECTOR STORE] Indexando {len(chunks)} fragmentos...")

    
    
    # 4. Enriquecimiento de Contexto
    for chunk in chunks:
        filename = chunk.metadata.get("source_filename", "Desconocido")
        chunk.page_content = f"Documento Fuente: {filename}\nContenido Legal:\n{chunk.page_content}"

    # 5. Ingesta usando el Cliente HTTP
    ids_list = await client.aadd_documents(
        documents=chunks
    )
    
    print(f"✅ [VECTOR STORE] Ingesta completada en el Servidor.")
    return ids_list



In [52]:
ids_list = await ingest_to_chroma(chunks= chunks, embedding_model= get_embedding_model(), reset_db= True)

[EMBEDDING] Conectando a Ollama en: http://localhost:11435
[EMBEDDING] Modelo seleccionado: qwen3-embedding:8b
[VECTOR STORE] Conectando a ChromaDB Server (localhost:8000)...
[VECTOR STORE] Eliminando colección 'kantuta_legal'...
[VECTOR STORE] Indexando 164 fragmentos...
✅ [VECTOR STORE] Ingesta completada en el Servidor.


In [53]:
len(chunks)

164